# ReportLens — OCR fine-tuning (TrOCR) on Colab

Fine-tunes `microsoft/trocr-small-printed` on **synthetic lab-report lines** and reports
Character Error Rate (CER). You only need to run the cells top-to-bottom.

**Before you start:** set the GPU runtime →  *Runtime → Change runtime type → Hardware
accelerator: **T4 GPU** → Save*.

Lower CER is better (0.0 = perfect). On ~800 synthetic reports and 3 epochs this typically
reaches a very low CER because the fonts/layout are consistent — that is expected and fine
for a portfolio demo. The interesting engineering is the synthetic-data + training pipeline.

## 1. Check the GPU is on

In [ ]:
!nvidia-smi

## 2. Clone the repo (always pulls the latest code)

Pulls in both `ocr_engine` and the `data_synthesis` package that generates the training
data. If the repo is already there from an earlier run, it is **reset to the latest
`main`** — otherwise a stale checkout would silently keep running old code.

The cell prints the commit it's running, so you can confirm you have the newest fixes.

In [ ]:
%cd /content
# Clone if missing, otherwise hard-reset to the latest main. Re-running the notebook in an
# existing session must NOT keep a stale checkout (that silently hides pushed fixes).
# This clone is a disposable training checkout - don't edit files here.
!if [ -d reportlens ]; then \
   cd reportlens && git fetch origin && git reset --hard origin/main; \
 else \
   git clone https://github.com/Satwiksingh123/reportlens-backend-.git reportlens; \
 fi
%cd /content/reportlens
!echo "--- running commit ---" && git log --oneline -1

## 3. Install the training dependencies (heavy — ~2-3 min)

Pins `transformers<5` (v5 changed tokenizer loading and breaks TrOCR) and installs
`sentencepiece`/`protobuf`, which TrOCR needs to build its tokenizer.

If pip prints a message about restarting the runtime, do *Runtime → Restart session*, then
re-run this cell and continue. Training itself runs in a fresh process, so it picks up the
right versions either way.

In [ ]:
!pip install -q -e "services/ocr_engine[train]"
!pip install -q pillow  # data_synthesis rendering
# TrOCR builds its fast tokenizer by converting the slow one, which needs sentencepiece
# + protobuf. Install explicitly so the conversion can't fail on a fresh Colab runtime.
!pip install -q sentencepiece protobuf

import transformers
print('transformers', transformers.__version__)
import sentencepiece
print('sentencepiece OK')

## 4. Train
Generates 800 synthetic reports, crops labelled lines, fine-tunes TrOCR for 3 epochs, then
prints the held-out CER and saves the model to `services/ocr_engine/artifacts/trocr-lab`.

First run downloads the base TrOCR weights (~300 MB). Full run is roughly 15-30 min on a T4.
Bump `--num-reports` / `--epochs` for a stronger model, lower them for a quick smoke test.

In [ ]:
%cd /content/reportlens/services/ocr_engine
!python -m ocr_engine.train_trocr --num-reports 800 --epochs 3 --batch-size 8 \
    --out artifacts/trocr-lab

## 5. Quick sanity check on one synthetic page
Runs the **full pipeline** (classical line segmentation + your fine-tuned recogniser) on a
freshly generated report and prints the recognised text.

In [ ]:
import sys; sys.path.insert(0, '../data_synthesis')
from data_synthesis.generator import generate_report, render_report
from ocr_engine.recognizer import TrOCRRecognizer
from ocr_engine.infer import extract_text_from_pil

report = generate_report(seed=9999)
img, _ = render_report(report, add_noise=True, seed=9999)
rec = TrOCRRecognizer(model_dir='artifacts/trocr-lab')
print(extract_text_from_pil(img, rec))

## 6. Download the fine-tuned model
Zips the model folder so you can save it (or upload it to the API host later). It is too
large for git — keep it as a release asset / on Drive, not in the repo.

In [ ]:
!cd artifacts && zip -qr trocr-lab.zip trocr-lab
from google.colab import files
files.download('artifacts/trocr-lab.zip')

### (Optional) Save to Google Drive instead
```python
from google.colab import drive; drive.mount('/content/drive')
!cp artifacts/trocr-lab.zip /content/drive/MyDrive/
```